# 01 — Create Folder Structure
**Day 2 | Part 2**

Creates all Bronze / Silver / Gold folders inside ADLS Gen2 containers.

**Prerequisites:**
- Secret scope `kv-ev-scope` exists (Day 1 Part 6.5)
- Secrets `adls-account-name`, `sp-client-id`, `sp-client-secret`, `sp-tenant-id` in Key Vault
- SP has `Storage Blob Data Contributor` role on `evdatalakedev`

**Cost: ₹0** — creating folders in ADLS Gen2 is free.

## Cell 1 — Configure storage auth (SP OAuth — no mount)

In [ ]:
# Load secrets from Key Vault — nothing hardcoded
SCOPE = "kv-ev-scope"

storage_account  = dbutils.secrets.get(scope=SCOPE, key="adls-account-name")
sp_client_id     = dbutils.secrets.get(scope=SCOPE, key="sp-client-id")
sp_client_secret = dbutils.secrets.get(scope=SCOPE, key="sp-client-secret")
sp_tenant_id     = dbutils.secrets.get(scope=SCOPE, key="sp-tenant-id")

# Configure Spark OAuth for this storage account
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", sp_client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", sp_client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token")

# Path helper — use instead of /mnt/ paths
def abfss(container: str, path: str = "") -> str:
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return f"{base}/{path}" if path else base

print(f"Storage account : {storage_account}")
print("OAuth config set — OK")
print("abfss() helper ready — OK")

## Cell 2 — Verify connection to all 4 containers

In [ ]:
print("=== Verifying access to all containers ===\n")
all_ok = True

for container in ["bronze", "silver", "gold", "source"]:
    try:
        items = dbutils.fs.ls(abfss(container))
        print(f"  {container:<8} OK — {len(items)} items")
    except Exception as e:
        print(f"  {container:<8} ERROR — {e}")
        all_ok = False

if all_ok:
    print("\nAll containers accessible — proceeding to create folders.")
else:
    print("\nFix errors above before continuing.")
    print("  403 Forbidden        → SP missing Storage Blob Data Contributor role")
    print("  AuthenticationFailed → re-check sp-client-id / sp-client-secret in Key Vault")
    raise Exception("Storage access failed — fix errors above")

## Cell 3 — Create all 40+ folders

In [ ]:
folders = [
    # Bronze — API (one folder per endpoint, 18 total)
    abfss("bronze", "api/payments"),
    abfss("bronze", "api/sessions"),
    abfss("bronze", "api/customers"),
    abfss("bronze", "api/fleet"),
    abfss("bronze", "api/chargers"),
    abfss("bronze", "api/vehicles"),
    abfss("bronze", "api/complaints"),
    abfss("bronze", "api/maintenance_events"),
    abfss("bronze", "api/energy_prices"),
    abfss("bronze", "api/tariffs"),
    abfss("bronze", "api/charge_cards"),
    abfss("bronze", "api/employees"),
    abfss("bronze", "api/partners"),
    abfss("bronze", "api/cities"),
    abfss("bronze", "api/stations"),
    abfss("bronze", "api/states"),
    abfss("bronze", "api/weather"),
    abfss("bronze", "api/pipeline_audit"),
    # Bronze — Blob file uploads
    abfss("bronze", "blob/iot_sessions"),
    abfss("bronze", "blob/maintenance"),
    abfss("bronze", "blob/invoices"),
    abfss("bronze", "blob/station_configs"),
    abfss("bronze", "blob/energy_reports"),
    # Bronze — Streaming + checkpoints
    abfss("bronze", "streaming/event_hub"),
    abfss("bronze", "_checkpoints"),
    # Silver — one Delta table per entity
    abfss("silver", "payments"),
    abfss("silver", "sessions"),
    abfss("silver", "customers"),
    abfss("silver", "fleet"),
    abfss("silver", "vehicles"),
    abfss("silver", "chargers"),
    abfss("silver", "complaints"),
    abfss("silver", "maintenance_events"),
    abfss("silver", "energy_prices"),
    abfss("silver", "pipeline_audit"),
    # Gold — aggregated tables
    abfss("gold", "fact_charging_sessions"),
    abfss("gold", "fact_payments"),
    abfss("gold", "dim_customer"),
    abfss("gold", "dim_vehicle"),
    abfss("gold", "dim_station"),
    abfss("gold", "agg_daily_revenue"),
    abfss("gold", "agg_station_utilization"),
]

created, errors = [], []

for folder in folders:
    try:
        dbutils.fs.mkdirs(folder)
        print(f"  Created : {folder}")
        created.append(folder)
    except Exception as e:
        print(f"  ERROR   : {folder} — {e}")
        errors.append(folder)

print(f"\nDone — {len(created)}/{len(folders)} folders created.")

if errors:
    print(f"\nFailed ({len(errors)}):")
    for f in errors:
        print(f"  {f}")
    print("\nCommon cause: SP missing Storage Blob Data Contributor role — check IAM on evdatalakedev.")
    raise Exception(f"{len(errors)} folder(s) failed")

## Cell 4 — Verify folder structure

In [ ]:
print("=== Bronze top-level ===")
for item in dbutils.fs.ls(abfss("bronze")):
    print(f"  {item.name}")

print("\n=== Bronze/api (should be 18 folders) ===")
api_folders = list(dbutils.fs.ls(abfss("bronze", "api")))
for item in api_folders:
    print(f"  {item.name}")
print(f"  Total: {len(api_folders)} folders")

print("\n=== Bronze/blob ===")
for item in dbutils.fs.ls(abfss("bronze", "blob")):
    print(f"  {item.name}")

print("\n=== Silver ===")
for item in dbutils.fs.ls(abfss("silver")):
    print(f"  {item.name}")

print("\n=== Gold ===")
for item in dbutils.fs.ls(abfss("gold")):
    print(f"  {item.name}")

print("\nFolder structure verified — proceed to Part 3.")